# Volatility on the new markets (added after looking)

Plan committed before any of these ranges was looked at: see [PLAN_clean_checks.md](PLAN_clean_checks.md).

- **Markets:** Korea, Taiwan, Hong Kong and Japan. Never used for volatility before.
- **Same as step 7:** predict the log of log(High / Low) for day t. Same as yesterday, HAR, HAR + US, GARCH, random forest and the GRU (seeds 0, 1, 2), with step 7's settings.
- **Test years 2017–2026:** train up to Y−2, tune on Y−1, test on Y. Run once.
- **Expected:** HAR beats same as yesterday and the forest everywhere, and GARCH in at least 3 markets. The GRU ties. HAR + US gains at most 0.03 in R².

In [ ]:
from functools import partial
from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from data import NEW_MARKETS, load_all, load_us
from evaluation import holm, paired_loss_test
from features import make_har, make_har_with_us, make_range_sequence
from gru import HIDDEN_SIZES
from more_inputs import TEST_YEARS
from volatility import fit_gru_vol, fit_regressor, pick_gru_vol_setting, qlike, walk_forward_garch, walk_forward_vol

markets = load_all(include_holdout=True, markets=NEW_MARKETS)
us = load_us(include_holdout=True)
RESULTS = Path("results") / "new_markets_volatility"  # not in git
RESULTS.mkdir(parents=True, exist_ok=True)

def run_market(path, run, df):
    """Load the saved predictions, or run the model once and save them."""
    if path.exists():
        return pd.read_pickle(path)
    preds = run(df)
    preds.to_pickle(path)
    return preds

def run_all_markets(name, run):
    runs = Parallel(n_jobs=2)(delayed(run_market)(RESULTS / f"{name}_{m}.pkl".replace(" ", "_"), run, df)
                              for m, df in markets.items())
    return dict(zip(markets, runs))

def vol(fit, settings, make_x, **options):
    return partial(walk_forward_vol, fit=fit, settings=settings, make_x=make_x, test_years=TEST_YEARS, **options)

## Run the models

About 40 minutes on a first run, mostly the GRU. Step 7's settings, unchanged.

In [ ]:
LINE = [LinearRegression()]
FOREST = [RandomForestRegressor(n_estimators=300, min_samples_leaf=leaf, random_state=0) for leaf in [10, 50, 200]]

har = run_all_markets("har", vol(fit_regressor, LINE, make_har))
har_us = run_all_markets("har_us", vol(fit_regressor, LINE, partial(make_har_with_us, us=us)))
garch = run_all_markets("garch", partial(walk_forward_garch, test_years=TEST_YEARS))
forest = run_all_markets("forest", vol(fit_regressor, FOREST, make_har))
gru = {seed: run_all_markets(f"gru_seed{seed}", vol(partial(fit_gru_vol, seed=seed), HIDDEN_SIZES, make_range_sequence,
                                                    pick=partial(pick_gru_vol_setting, seed=seed)))
       for seed in [0, 1, 2]}

## Scores, all ten test years together

- **R²:** 1 is perfect, 0 is no better than the average of all earlier days.
- **QLIKE:** how far off in relative terms, like "2× too high". Lower is better.
- **p vs HAR:** the paired test on the same days, Holm-adjusted over the 24 comparisons for each score (6 rivals × 4 markets).

In [ ]:
same = {m: p.assign(predicted=p["same_as_yesterday"]) for m, p in har.items()}
MODELS = {"same as yesterday": same, "HAR": har, "HAR + US": har_us, "GARCH": garch, "forest": forest,
          "GRU 0": gru[0], "GRU 1": gru[1], "GRU 2": gru[2]}

def common_days(m):
    days = MODELS["HAR"][m].index
    for runs in MODELS.values():
        days = days.intersection(runs[m].index)
    return days

days = {m: common_days(m) for m in markets}
print("days scored:", {m: len(d) for m, d in days.items()}, "| HAR days left out:", {m: len(har[m]) - len(d) for m, d in days.items()})

def r2(p):
    error, error_of_average = p["predicted"] - p["actual"], p["average_so_far"] - p["actual"]
    return 1 - (error ** 2).sum() / (error_of_average ** 2).sum()

def squared_error(p):
    return (p["predicted"] - p["actual"]) ** 2

def table(score):
    return pd.DataFrame({name: {m: score(runs[m].loc[days[m]]) for m in markets} for name, runs in MODELS.items()})

def vs_har(loss):
    """Each rival's p-value against HAR, then Holm over the whole table."""
    p = pd.DataFrame({name: {m: paired_loss_test(loss(runs[m].loc[days[m]]), loss(har[m].loc[days[m]])) for m in markets}
                      for name, runs in MODELS.items() if name != "HAR"})
    return holm(p.stack()).unstack()[p.columns]

for title, t in {"R2": table(r2), "p vs HAR on squared errors (Holm)": vs_har(squared_error),
                 "QLIKE": table(lambda p: qlike(p).mean()), "p vs HAR on QLIKE (Holm)": vs_har(qlike)}.items():
    print(title)
    display(t.round(3))

HAR's R² in each test year.

In [ ]:
pd.DataFrame({m: {year: r2(f) for year, f in p.loc[days[m]].groupby("year")} for m, p in har.items()}).T.round(2)

CONCLUSIONS